# 提案 conf スイープ プローブ — fold0(学習なし、既存ckptの使い回し)

v57(§6.16)で dense2x は dense FN を減らしたが FP 爆増で refined PQ が悪化した。
**conf 閾値を動かして「FN改善を残しつつFPを削る」点がないか**を、学習済みckpt
5本(prod / v57baseline×2seed / v57dense2x×2seed)に対して評価する。
副次的に現行の操作点 CONF=0.26 が妥当かも確認できる。

| 列 | 内容 |
|---|---|
| raw PQ | 生YOLO proposals → ownership(refinerなし) |
| refined PQ | proposals → 既存標準refiner → shrink@0.45 → ownership(確定構成) |
| FN(dense/sparse) | 検証画像を train q75 cut でスライスした未マッチGT数 |

fold0基準: refined PQ 0.4393。**注意: これは評価fold上の探索であり、
ここで選んだ閾値をそのまま採用しない(fold0選択バイアスの前科あり)。有望点は
fold1/2での独立確認が必須。** CPU 契約を検証した補修版で、GPU受入・操作点採用は別途必要。
旧manifestは入力契約が無ければ再利用せず、履歴を保持して新しい保存先を選ぶ。
fold1/2 では CFG.SELECTED_CONF に fold0 で事前選択した値を指定する。未指定なら評価前に停止する。
GPU必須、学習なしで合計1〜1.5時間見込み。


In [ ]:
# [1/6] Drive接続、設定、前提の確認
!pip -q install ultralytics pycocotools
import hashlib, json, math, random, shutil, sys, time, warnings
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

class CFG:
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v58_confsweep")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    FOLD = 0
    PROTOCOL_SHA256 = '6b6f7362888ee094a02c4adffa6c7cb2252be4e41878c381a920acb9d028ece4'
    # --- OOF評価は確定構成(build_v56/v57系と同一) ---
    YOLO_IMGSZ = 1280
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    PREDICT_CONF_FLOOR = 0.03
    PREDICT_IOU = 0.55
    CONF_SWEEP = (0.05, 0.08, 0.12, 0.16, 0.2, 0.26, 0.32, 0.4)
    SELECTED_CONF = None  # fold1/2: set the single threshold selected on fold0 before running
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    BASELINE_PQ_REFINED = 0.4393
    DENSE_QUANTILE = 0.75
    MANIFEST = DRIVE_ROOT / "artifacts" / "v58_conf_sweep_fold0_manifest.json"

def confirmation_confidences():
    if CFG.FOLD == 0:
        return CFG.CONF_SWEEP
    value = CFG.SELECTED_CONF
    if type(value) not in (int, float) or not math.isfinite(value) or not CFG.PREDICT_CONF_FLOOR <= value <= 1:
        raise ValueError("Set SELECTED_CONF to the single threshold preselected on fold0; confirmation folds cannot sweep")
    return (value,)

CFG.CONF_SWEEP = confirmation_confidences()

def checkpoint_map(fold):
    """label -> best.pt。存在しないものは呼び出し側でskipする。"""
    runs = CFG.DRIVE_ROOT / "runs"
    return {
        "prod": runs / f"yolo_fold_{fold}" / "weights" / "best.pt",
        "v57baseline_s3407": runs / "v57_dense_oversample" / f"yolo_v57_baseline_s3407_fold{fold}" / "weights" / "best.pt",
        "v57baseline_s8407": runs / "v57_dense_oversample" / f"yolo_v57_baseline_s8407_fold{fold}" / "weights" / "best.pt",
        "v57dense2x_s3407": runs / "v57_dense_oversample" / f"yolo_v57_dense2x_s3407_fold{fold}" / "weights" / "best.pt",
        "v57dense2x_s8407": runs / "v57_dense_oversample" / f"yolo_v57_dense2x_s8407_fold{fold}" / "weights" / "best.pt",
    }

for path in [CFG.LOCAL_ROOT, CFG.PREPROC_DIR]:
    path.mkdir(parents=True, exist_ok=True)

required = {
    "competition data": CFG.DATA_CACHE,
    "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
    "前処理画像キャッシュ": CFG.PREPROC_CACHE,
    f"fold{CFG.FOLD} 標準refiner": CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required})
RUNTIME_INFO = {"started_utc": datetime.now(timezone.utc).isoformat(),
                "service": "Google Colab", "project": f"solar_v58_fold{CFG.FOLD}",
                "GPU": torch.cuda.get_device_name(0), "execution_scope": "inference_only",
                "compute_units": None,
                "compute_units_unavailable_reason": "Colab runtime API does not expose billing usage"}
print(RUNTIME_INFO)


In [ ]:
# [2/6] COCOとfoldを読み、密度スライス用のcutを計算し、前処理画像をローカルSSDへ展開
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"学習annotation JSONが見つかりません: {base}")
    return matches[0].parents[2]

TRAIN_JSON = next(find_dataset_root(CFG.DATA_CACHE).rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
with open(TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)

images = {item["id"]: item for item in coco["images"]}
annotations_by_image = defaultdict(list)
for ann in coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

fold_frame = pd.read_csv(CFG.ARTIFACT_ROOT / "grouped_folds.csv")
if fold_frame["file_name"].duplicated().any() or not fold_frame["fold"].isin([0, 1, 2]).all():
    raise ValueError("grouped folds must have unique filenames and integer folds 0/1/2")
fold_of_file = dict(zip(fold_frame["file_name"], fold_frame["fold"].astype(int)))
if any(name not in fold_of_file for name in train_files):
    raise ValueError("Observations missing from grouped folds")
train_files_fold = [n for n in train_files if fold_of_file[n] != CFG.FOLD]
valid_files_fold = [n for n in train_files if fold_of_file[n] == CFG.FOLD]

def noncrowd_count(image_id):
    return sum(1 for a in annotations_by_image[image_id] if not a.get("iscrowd", 0))

# v57と同じ定義: cutは学習サンプル(file,annotator)のGT数q75
train_sample_counts = [
    noncrowd_count(image_id)
    for name in train_files_fold
    for image_id in file_to_ids[name]
]
if not train_sample_counts or not valid_files_fold:
    raise ValueError("Both training and validation observations are required")
DENSE_CUT = float(np.quantile(train_sample_counts, CFG.DENSE_QUANTILE))
val_density = {
    name: float(np.mean([noncrowd_count(i) for i in file_to_ids[name]]))
    for name in valid_files_fold
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

preprocessed_hashes = {}
for name in tqdm(valid_files_fold, desc="前処理画像をSSDへ・SHA256"):
    local = CFG.PREPROC_DIR / name
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CFG.PREPROC_CACHE / name, local)
    preprocessed_hashes[name] = file_sha256(local)

PILOT_CONTRACT = {
    "schema_version": 1, "fold": CFG.FOLD, "protocol_sha256": CFG.PROTOCOL_SHA256,
    "annotations_sha256": file_sha256(TRAIN_JSON),
    "folds_sha256": file_sha256(CFG.ARTIFACT_ROOT / "grouped_folds.csv"),
    "refiner_sha256": file_sha256(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt"),
    "checkpoint_sha256": {label: file_sha256(path.resolve(strict=True))
                          for label, path in checkpoint_map(CFG.FOLD).items() if path.is_file()},
    "preprocessed_sha256": hashlib.sha256(
        json.dumps(preprocessed_hashes, sort_keys=True).encode("utf-8")).hexdigest(),
    "dense_quantile": CFG.DENSE_QUANTILE, "dense_cut": DENSE_CUT,
    "conf_sweep": list(CFG.CONF_SWEEP),
    "recipe": {name: getattr(CFG, name) for name in (
        "YOLO_IMGSZ", "CROP_SIZE", "CROP_CONTEXT", "PREDICT_CONF_FLOOR", "PREDICT_IOU",
        "REFINER_THRESHOLD", "DROP_THRESHOLD", "MIN_AREA")},
    "runtime_versions": {"python": sys.version, "torch": torch.__version__,
                         "ultralytics": version("ultralytics"), "numpy": np.__version__,
                         "opencv": cv2.__version__, "pycocotools": version("pycocotools")},
}

def require_checkpoint(path, expected):
    if file_sha256(path) != expected:
        raise ValueError(f"{path}: checkpoint changed after input contract; restart with fresh inputs")

def save_probe_manifest(manifest):
    import os
    import tempfile
    payload = json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=CFG.MANIFEST.parent,
                                         suffix=".tmp", delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, CFG.MANIFEST)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def read_probe_manifest():
    if not CFG.MANIFEST.exists():
        return {"experiment_id": f"v58_conf_sweep_fold{CFG.FOLD}",
                "input_contract": PILOT_CONTRACT, "runs": []}
    manifest = json.loads(CFG.MANIFEST.read_text(encoding="utf-8"))
    if (manifest.get("input_contract") != PILOT_CONTRACT
            or manifest.get("experiment_id") != f"v58_conf_sweep_fold{CFG.FOLD}"):
        raise ValueError("Manifest input contract differs or is absent; retain it and choose a fresh manifest path")
    known = set(PILOT_CONTRACT["checkpoint_sha256"])
    seen = set()
    if not isinstance(manifest.get("runs"), list):
        raise ValueError("Manifest rows must be a list")
    for row in manifest["runs"]:
        if not isinstance(row, dict):
            raise ValueError("Invalid manifest row")
        key = (row.get("ckpt"), row.get("conf"))
        if (key in seen or key[0] not in known or key[1] not in CFG.CONF_SWEEP
                or row.get("fold") != CFG.FOLD):
            raise ValueError("Invalid or duplicate manifest row")
        path = row.get("checkpoint")
        if not isinstance(path, str) or Path(path).resolve() != checkpoint_map(CFG.FOLD)[key[0]].resolve():
            raise ValueError("Manifest row checkpoint path differs from its label")
        for mode in ("raw", "refined"):
            if not isinstance(row.get(mode), dict):
                raise ValueError("Manifest row has invalid metrics")
            pq = row[mode].get("pq")
            if not isinstance(pq, (int, float)) or not math.isfinite(pq) or not 0 <= pq <= 1:
                raise ValueError("Manifest row has invalid PQ")
        counts = [row.get("fn_dense_val"), row.get("fn_sparse_val"),
                  row["refined"].get("fp"), row["refined"].get("fn")]
        if any(type(count) is not int or count < 0 for count in counts):
            raise ValueError("Manifest row must contain nonnegative integer FN/FP counts")
        seen.add(key)
    return manifest

# Fail before loading/refining GPU models or changing the historical manifest.
read_probe_manifest()

print({"検証(OOF)": len(valid_files_fold), "dense_cut": DENSE_CUT,
       "dense検証画像数": int(sum(v >= DENSE_CUT for v in val_density.values()))})


In [ ]:
# [3/6] MediSegUNet (solar_filament/models/mediseg_models.py と同一。標準refinerの評価に使用)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


In [ ]:
# [4/6] 推論・後処理・採点関数(build_v57系と同一経路。提案ごとのrefinedは1回だけ計算してRLEで保持)
from pycocotools import mask as mask_utils
from ultralytics import YOLO

require_checkpoint(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
                   PILOT_CONTRACT["refiner_sha256"])
refiner = MediSegUNet(3, 1, False).cuda().eval()
_ckpt = torch.load(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
                    map_location="cuda", weights_only=False)
refiner.load_state_dict(_ckpt.get("best_model") or _ckpt["model"], strict=True)
del _ckpt

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return output, confidences

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposal(image, proposal):
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= CFG.REFINER_THRESHOLD
    return refined & proposal  # shrink(確定構成)

def encode_one(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

def decode_one(rle):
    return mask_utils.decode(rle) != 0

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

class PanopticQuality:
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.tp_ious = []
        self.fp_count = 0
        self.fn_count = 0
    def add_matrix(self, matrix, n_gt, n_pred):
        if n_gt == 0:
            self.fp_count += n_pred; return
        if n_pred == 0:
            self.fn_count += n_gt; return
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        self.tp_ious.extend(scores)
        self.fp_count += int((hits.sum(axis=0) == 0).sum())
        self.fn_count += int((hits.sum(axis=1) == 0).sum())
    @property
    def denominator(self):
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count
    @property
    def score(self):
        d = self.denominator
        return float(sum(self.tp_ious) / d) if d > 0 else 0.0
    def sq_rq(self):
        tp = len(self.tp_ious)
        sq = float(sum(self.tp_ious) / tp) if tp > 0 else 0.0
        rq = float(tp / (tp + 0.5 * self.fp_count + 0.5 * self.fn_count)) if (tp + self.fp_count + self.fn_count) > 0 else 0.0
        return sq, rq
    def summary(self):
        sq, rq = self.sq_rq()
        return {"pq": self.score, "sq": sq, "rq": rq, "tp": len(self.tp_ious),
                "fp": self.fp_count, "fn": self.fn_count, "denominator": self.denominator}

gt_rles = {}
for image_id, info in images.items():
    height, width = int(info["height"]), int(info["width"])
    gt_rles[image_id] = [to_rle(ann, height, width) for ann in annotations_by_image[image_id] if not ann.get("iscrowd", 0)]

print("後処理・採点関数の定義OK")


In [ ]:
# [5/6] ckptごとに「floor推論→全提案をrefineしてRLE保持」→confスイープはメモリ上で評価
def collect_cache(best_path, label):
    """floor(conf=0.03)で全検証画像をpredictし、raw/refined両方のRLE+score+面積を保持。"""
    if label not in PILOT_CONTRACT["checkpoint_sha256"] or best_path.resolve() != checkpoint_map(CFG.FOLD)[label].resolve():
        raise ValueError("Evaluation checkpoint label/path differs from input contract")
    require_checkpoint(best_path, PILOT_CONTRACT["checkpoint_sha256"][label])
    yolo_model = YOLO(str(best_path))
    cache = []
    for name in tqdm(valid_files_fold, desc=f"{label} fold{CFG.FOLD} 推論+refine"):
        image_path = CFG.PREPROC_DIR / name
        result = yolo_model.predict(str(image_path), imgsz=CFG.YOLO_IMGSZ,
                                     conf=CFG.PREDICT_CONF_FLOOR, iou=CFG.PREDICT_IOU,
                                     agnostic_nms=True, retina_masks=True, verbose=False)[0]
        masks, confidences = result_masks(result)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        entry = {"name": name, "scores": confidences, "areas": [], "raw": [], "refined": []}
        for mask in masks:
            entry["areas"].append(float(mask.sum()))
            entry["raw"].append(encode_one(mask))
            entry["refined"].append(encode_one(refine_proposal(image, mask)))
        entry["areas"] = np.asarray(entry["areas"], dtype=np.float64)
        cache.append(entry)
    del yolo_model
    torch.cuda.empty_cache()
    return cache

def score_conf(cache, conf):
    """保持済みRLEを conf で絞り、ownership→PQ を raw/refined 両方で採点 + 密度別FN。"""
    meters = {"raw": PanopticQuality(), "refined": PanopticQuality()}
    fn_dense = fn_sparse = 0
    for entry in cache:
        keep = [i for i in range(len(entry["scores"]))
                if entry["scores"][i] >= conf and entry["areas"][i] >= CFG.MIN_AREA]
        scores = entry["scores"][keep] if keep else np.zeros(0, np.float32)
        name = entry["name"]
        pred_rles_by_mode = {}
        for mode in ("raw", "refined"):
            masks = [decode_one(entry[mode][i]) for i in keep]
            preds = apply_ownership(masks, scores)
            pred_rles = []
            if preds:
                encoded = mask_utils.encode(np.asfortranarray(np.stack(preds, axis=-1).astype(np.uint8)))
                pred_rles = encoded if isinstance(encoded, list) else [encoded]
            pred_rles_by_mode[mode] = pred_rles
        file_fn = 0
        for image_id in file_to_ids[name]:
            truth = gt_rles[image_id]
            for mode in ("raw", "refined"):
                matrix = iou_matrix(truth, pred_rles_by_mode[mode])
                meters[mode].add_matrix(matrix, len(truth), len(pred_rles_by_mode[mode]))
                if mode == "refined":
                    if matrix.size:
                        file_fn += int((matrix.max(axis=1) <= 0.5).sum())
                    else:
                        file_fn += len(truth)
        if val_density[name] >= DENSE_CUT:
            fn_dense += file_fn
        else:
            fn_sparse += file_fn
    return {"raw": meters["raw"].summary(), "refined": meters["refined"].summary(),
            "fn_dense_val": fn_dense, "fn_sparse_val": fn_sparse}


In [ ]:
# [6/6] ckpt×confを直列実行、runごとにmanifestへ逐次追記(途中再開対応)
manifest = read_probe_manifest()
rows = manifest["runs"]
done_keys = {(r["ckpt"], r["conf"]) for r in rows}
available = {label: p for label, p in checkpoint_map(CFG.FOLD).items() if p.is_file()}
if not available:
    raise FileNotFoundError("No evaluation checkpoint available")
missing_labels = sorted(set(checkpoint_map(CFG.FOLD)) - set(available))
manifest["missing_checkpoints"] = missing_labels
manifest["status"] = "partial_checkpoint_set" if missing_labels else "in_progress"
manifest["runtime"] = {**RUNTIME_INFO,
                       "execution_scope": "inference_only", "completion": "in_progress"}
session_started = time.time()
torch.cuda.reset_peak_memory_stats()
for label, path in checkpoint_map(CFG.FOLD).items():
    if not path.exists():
        print(f"{label}: ckptなし({path}) — skip")

try:
    for label, best_path in available.items():
        pending = [c for c in CFG.CONF_SWEEP if (label, c) not in done_keys]
        if not pending:
            print(f"{label}: manifest済み、skip")
            continue
        started = time.time()
        cache = collect_cache(best_path, label)  # 残っているconfがある限り再収集
        for conf in CFG.CONF_SWEEP:
            if (label, conf) in done_keys:
                continue
            metrics = score_conf(cache, conf)
            row = {"ckpt": label, "conf": conf, "fold": CFG.FOLD,
                   "checkpoint": str(best_path), **metrics}
            rows.append(row)
            done_keys.add((label, conf))
            manifest["runs"] = rows
            manifest["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
            save_probe_manifest(manifest)
            print(json.dumps(row, ensure_ascii=False))
        del cache
        torch.cuda.empty_cache()
        print(f"{label}: 完了({round((time.time() - started) / 60, 1)}min)")
except (Exception, KeyboardInterrupt) as error:
    manifest["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
    try:
        peak_vram = int(torch.cuda.max_memory_allocated())
        peak_note = None
    except Exception as stats_error:
        peak_vram = None
        peak_note = f"CUDA memory statistics unavailable: {type(stats_error).__name__}"
    manifest["runtime"].update({"duration_seconds": round(time.time() - session_started, 3),
                                "duration_scope": "current_inference_session",
                                "peak_vram_bytes": peak_vram,
                                "peak_vram_unavailable_reason": peak_note,
                                "completion": manifest["status"], "error_type": type(error).__name__})
    save_probe_manifest(manifest)
    raise
manifest["status"] = "partial_checkpoint_set" if missing_labels else "complete"
manifest["runtime"].update({"duration_seconds": round(time.time() - session_started, 3),
                            "duration_scope": "current_inference_session",
                            "peak_vram_bytes": int(torch.cuda.max_memory_allocated()),
                            "completion": "completed_available_checkpoints"})
save_probe_manifest(manifest)

print(f"\n保存: {CFG.MANIFEST}")

# --- 集計: conf×refined PQ の曲線 + paired 比較 ---
print("\n=== refined PQ 曲線(ckpt別) ===")
for label in available:
    series = sorted((r["conf"], r["refined"]["pq"], r["raw"]["pq"],
                     r["fn_dense_val"], r["fn_sparse_val"],
                     r["refined"]["fp"], r["refined"]["fn"])
                    for r in rows if r["ckpt"] == label)
    for conf, pq, raw_pq, fn_d, fn_s, fp, fn in series:
        print(f"{label} conf={conf:.2f}: refined {pq:.4f} | raw {raw_pq:.4f} | "
              f"FP {fp} FN {fn} | FN(dense/sparse) {fn_d}/{fn_s}")

print("\n=== paired Δ(v57dense2x - v57baseline、同seed・同conf) ===")
for seed in (3407, 8407):
    d_label, b_label = f"v57dense2x_s{seed}", f"v57baseline_s{seed}"
    for conf in CFG.CONF_SWEEP:
        d = next((r for r in rows if r["ckpt"] == d_label and r["conf"] == conf), None)
        b = next((r for r in rows if r["ckpt"] == b_label and r["conf"] == conf), None)
        if d and b:
            print(f"seed {seed} conf={conf:.2f}: Δrefined {d['refined']['pq'] - b['refined']['pq']:+.4f} | "
                  f"ΔFN(dense) {d['fn_dense_val'] - b['fn_dense_val']:+d} | "
                  f"ΔFP {d['refined']['fp'] - b['refined']['fp']:+d}")

print(f"\n採否の目安: fold0 のみ探索、fold1/2 は事前選択した単一confの確認。dense2xが「両seed・同じconfで」"
      "baselineを上回る点があれば、そのconfだけをfold1/2で独立確認(v57のfold1/2 ckptが必要)。"
      "prodの曲線は現行CONF=0.26が妥当かの健全性確認として使う。")
